In [2]:
!pip install pytorch_metric_learning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 9.0 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 12.0 MB/s  0:00:40m0:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 13.2 MB/s  0:00:40m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 9.9 MB/s  0:00:176m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 20.2 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 19.3 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.9/404.9 MB 14.6 MB/s  0:00:25m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 17.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 MB 10.9 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 5.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 MB 11.0 MB/s  0:00:19m0:00:0100:01
   ━━━━━━━━━━━

In [4]:
import os
import copy
import csv
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

from torch.utils.data import DataLoader, Subset, WeightedRandomSampler

from sklearn.metrics import roc_auc_score, roc_curve

from pytorch_metric_learning import miners, losses

import matplotlib.pyplot as plt

try:
    import umap
    UMAP_AVAILABLE = True
except Exception:
    UMAP_AVAILABLE = False


# ==========================================================
# CIFAR normalization constants
# ==========================================================
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)


# ==========================================================
# CONFIG
# ==========================================================
class Config:
    seed = 42
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Set True if you need strict reproducibility and can accept slower training.
    deterministic = False

    # ------------------------------
    # Data
    # ------------------------------
    data_root = "./data"
    image_size = 32

    # In-distribution classes from CIFAR-10
    id_classes = [0, 1, 2, 3, 4]

    # Real OOD dataset:
    # - "svhn"      : Street View House Numbers, external OOD, easy but real
    # - "cifar100"  : external dataset, harder than SVHN, still small images
    # - "textures"  : DTD textures, if available in your torchvision version
    ood_dataset = "svhn"

    # Limit OOD test samples for faster evaluation.
    # Set None or 0 to use all OOD test samples.
    ood_test_limit = 5000

    # ------------------------------
    # Experiments
    # ------------------------------
    # Supported experiment names:
    #
    #   ce
    #   triplet
    #   ms
    #   supcon
    #
    #   ce_outliers
    #   triplet_outliers
    #   ms_outliers
    #   supcon_outliers
    #
    # For very limited compute, use:
    #   experiments = ["ce", "triplet_outliers"]
    #
    # For a fuller comparison:
    #   experiments = ["ce", "triplet", "supcon", "triplet_outliers"]
    experiments = [
        "ce",
        "triplet",
        "supcon",
        "triplet_outliers",
    ]

    # ------------------------------
    # Training
    # ------------------------------
    epochs = 20
    batch_size = 128

    # Number of synthetic outlier images used per training batch
    # when the experiment name ends with "_outliers".
    outlier_batch_size = 64

    num_workers = 2

    lr = 0.05
    weight_decay = 5e-4

    # Balanced sampling helps metric-learning losses.
    use_balanced_sampler = True

    # Metric-loss weights.
    metric_weights = {
        "triplet": 0.10,
        "ms": 0.05,
        "supcon": 0.10,
    }

    # Triplet loss settings
    triplet_margin = 0.30

    # MultiSimilarityLoss settings
    ms_alpha = 2.0
    ms_beta = 50.0
    ms_base = 0.5

    # SupConLoss settings
    supcon_temperature = 0.10

    # Synthetic outlier training weight
    outlier_weight = 0.05

    # Temperature used for energy-based losses/scores.
    energy_temperature = 1.0

    # Save model checkpoints
    save_models = True

    # ------------------------------
    # Evaluation
    # ------------------------------
    eval_batch_size = 256

    # Five OOD scoring baselines.
    # Remove any method you do not want to run.
    score_methods = [
        "MSP",
        "Energy",
        "Prototype",
        "Mahalanobis",
        "kNN",
    ]

    knn_k = 10
    mahalanobis_reg = 1e-3

    # ------------------------------
    # UMAP
    # ------------------------------
    run_umap = True

    # Which experiments to visualize with UMAP.
    umap_experiments = [
        "ce",
        "triplet",
        "ms",
        "supcon",
        "triplet_outliers",
        "cs_outliers",
        "ms_outliers",
        "supcon_outliers"
    ]

    # Subsample UMAP for limited compute.
    umap_max_points = 2500
    umap_n_epochs = 200

    # ------------------------------
    # Output
    # ------------------------------
    output_dir = "results"


config = Config()


# ==========================================================
# SEEDING
# ==========================================================
def set_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = config.deterministic
    torch.backends.cudnn.benchmark = not config.deterministic


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


# ==========================================================
# MODEL
# ==========================================================
class ResNetEmbedder(nn.Module):
    """
    CIFAR-friendly ResNet-18 encoder + 5-class classifier.
    Returns normalized embeddings when return_emb=True.
    """

    def __init__(self, num_classes=5):
        super().__init__()

        base = models.resnet18(weights=None)

        # CIFAR-friendly stem:
        # Standard ImageNet ResNet uses 7x7 conv stride 2 + maxpool,
        # which downsamples 32x32 CIFAR images too aggressively.
        base.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        base.maxpool = nn.Identity()

        # Remove final FC layer; use everything before it as feature extractor.
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.embedding_dim = base.fc.in_features

        self.classifier = nn.Linear(self.embedding_dim, num_classes)

    def forward(self, x, return_emb=False):
        emb = self.features(x)
        emb = emb.view(emb.size(0), -1)

        logits = self.classifier(emb)

        if return_emb:
            emb_norm = F.normalize(emb, p=2, dim=1)
            return logits, emb_norm

        return logits


# ==========================================================
# DATASET HELPERS
# ==========================================================
class RemapDataset(torch.utils.data.Dataset):
    """
    Remaps original CIFAR labels to contiguous ID labels.
    Example:
        id_classes = [0,1,2,3,4]
        original labels stay [0,1,2,3,4] here,
        but this makes the code robust for non-contiguous ID classes.
    """

    def __init__(self, subset, class_map):
        self.subset = subset
        self.class_map = class_map

        self.labels = []
        for idx in subset.indices:
            original_label = int(subset.dataset.targets[idx])
            self.labels.append(int(class_map[original_label]))

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, target = self.subset[idx]
        target = int(target)
        return img, self.class_map.get(target, target)


def make_subset(dataset, classes, class_map=None):
    targets = np.array(dataset.targets)
    indices = np.where(np.isin(targets, classes))[0].tolist()
    subset = Subset(dataset, indices)

    if class_map is not None:
        return RemapDataset(subset, class_map)

    return subset


def maybe_limit_dataset(dataset, max_samples):
    if max_samples is None or max_samples <= 0:
        return dataset

    n = len(dataset)
    if n <= max_samples:
        return dataset

    rng = np.random.RandomState(config.seed)
    indices = rng.choice(n, size=max_samples, replace=False).tolist()
    return Subset(dataset, indices)


def get_ood_dataset(name, transform):
    name = name.lower()

    if name == "svhn":
        return torchvision.datasets.SVHN(
            root=config.data_root,
            split="test",
            download=True,
            transform=transform,
        )

    if name == "cifar100":
        return torchvision.datasets.CIFAR100(
            root=config.data_root,
            train=False,
            download=True,
            transform=transform,
        )

    if name in ["textures", "dtd"]:
        # DTD may not be available in older torchvision versions.
        return torchvision.datasets.DTD(
            root=config.data_root,
            split="test",
            download=True,
            transform=transform,
        )

    raise ValueError(f"Unknown OOD dataset: {name}")


def build_datasets():
    train_transform = transforms.Compose([
        transforms.Resize((config.image_size, config.image_size)),
        transforms.RandomCrop(config.image_size, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    test_transform = transforms.Compose([
        transforms.Resize((config.image_size, config.image_size)),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])

    # Augmented training set
    trainset_aug = torchvision.datasets.CIFAR10(
        root=config.data_root,
        train=True,
        download=True,
        transform=train_transform,
    )

    # Clean training set for prototype / Mahalanobis / kNN fitting
    trainset_clean = torchvision.datasets.CIFAR10(
        root=config.data_root,
        train=True,
        download=True,
        transform=test_transform,
    )

    # ID test set
    testset = torchvision.datasets.CIFAR10(
        root=config.data_root,
        train=False,
        download=True,
        transform=test_transform,
    )

    id_class_map = {c: i for i, c in enumerate(config.id_classes)}

    id_train_aug = make_subset(trainset_aug, config.id_classes, id_class_map)
    id_train_clean = make_subset(trainset_clean, config.id_classes, id_class_map)
    id_test = make_subset(testset, config.id_classes, id_class_map)

    ood_name = config.ood_dataset.lower()

    try:
        ood_test = get_ood_dataset(ood_name, test_transform)
    except Exception as e:
        print(f"WARNING: Could not load OOD dataset '{ood_name}': {e}")
        print("Falling back to SVHN.")
        ood_name = "svhn"
        ood_test = get_ood_dataset(ood_name, test_transform)

    ood_test = maybe_limit_dataset(ood_test, config.ood_test_limit)

    return id_train_aug, id_train_clean, id_test, ood_test, ood_name


# ==========================================================
# DATALOADERS
# ==========================================================
def make_train_loader(dataset):
    if config.use_balanced_sampler:
        labels = np.asarray(dataset.labels)
        counts = np.bincount(labels, minlength=len(config.id_classes)).astype(np.float64)
        class_weights = 1.0 / np.maximum(counts, 1.0)
        sample_weights = class_weights[labels]

        sampler = WeightedRandomSampler(
            weights=torch.from_numpy(sample_weights).float(),
            num_samples=len(dataset),
            replacement=True,
        )

        return DataLoader(
            dataset,
            batch_size=config.batch_size,
            sampler=sampler,
            num_workers=config.num_workers,
            worker_init_fn=seed_worker,
            pin_memory=torch.cuda.is_available(),
        )

    generator = torch.Generator()
    generator.manual_seed(config.seed)

    return DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        worker_init_fn=seed_worker,
        generator=generator,
        pin_memory=torch.cuda.is_available(),
    )


def make_eval_loader(dataset):
    return DataLoader(
        dataset,
        batch_size=config.eval_batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        pin_memory=torch.cuda.is_available(),
    )


# ==========================================================
# CSV LOGGER
# ==========================================================
class CSVLogger:
    def __init__(self, filename, header):
        self.file = open(filename, "w", newline="")
        self.writer = csv.writer(self.file)
        self.writer.writerow(header)

    def log(self, row):
        self.writer.writerow(row)
        self.file.flush()

    def close(self):
        self.file.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()
        return False


# ==========================================================
# EXPERIMENT PARSING / METRIC LOSSES
# ==========================================================
def parse_experiment(exp_name):
    use_outliers = exp_name.endswith("_outliers")

    if use_outliers:
        base = exp_name[: -len("_outliers")]
    else:
        base = exp_name

    valid_bases = {"ce", "triplet", "ms", "supcon"}

    if base not in valid_bases:
        raise ValueError(
            f"Unknown experiment base '{base}'. "
            f"Valid bases: {sorted(valid_bases)}"
        )

    return base, use_outliers


def build_metric_loss(base):
    if base == "triplet":
        miner = miners.BatchHardMiner()
        loss_fn = losses.TripletMarginLoss(margin=config.triplet_margin)
        return miner, loss_fn

    if base == "ms":
        loss_fn = losses.MultiSimilarityLoss(
            alpha=config.ms_alpha,
            beta=config.ms_beta,
            base=config.ms_base,
        )
        return None, loss_fn

    if base == "supcon":
        if not hasattr(losses, "SupConLoss"):
            raise ValueError("SupConLoss is not available in this pytorch_metric_learning version.")

        loss_fn = losses.SupConLoss(temperature=config.supcon_temperature)
        return None, loss_fn

    return None, None


# ==========================================================
# SYNTHETIC OUTLIER HELPERS
# ==========================================================
def generate_synthetic_outliers(batch_size, device):
    """
    Simple synthetic outliers:
    random RGB images normalized with CIFAR statistics.

    This is intentionally simple and lightweight.
    """
    x = torch.rand(
        batch_size,
        3,
        config.image_size,
        config.image_size,
        device=device,
    )

    mean = torch.tensor(CIFAR_MEAN, dtype=torch.float32, device=device).view(1, 3, 1, 1)
    std = torch.tensor(CIFAR_STD, dtype=torch.float32, device=device).view(1, 3, 1, 1)

    return (x - mean) / std


def energy_outlier_loss(id_logits, ood_logits, T=1.0):
    """
    Energy-based outlier exposure loss.

    For ID data we want low energy:
        E_id = -logsumexp(logits_id / T)

    For synthetic OOD data we want high energy:
        E_ood = -logsumexp(logits_ood / T)

    Softplus formulation encourages:
        E_id  -> negative / low
        E_ood -> positive / high
    """
    e_id = -torch.logsumexp(id_logits / T, dim=1)
    e_ood = -torch.logsumexp(ood_logits / T, dim=1)

    loss_id = F.softplus(e_id).mean()
    loss_ood = F.softplus(-e_ood).mean()

    return loss_id + loss_ood


# ==========================================================
# TRAINING
# ==========================================================
def train_one_experiment(exp_name, init_state, id_train_set):
    set_seeds(config.seed)

    base, use_outliers = parse_experiment(exp_name)

    model = ResNetEmbedder(num_classes=len(config.id_classes)).to(config.device)
    model.load_state_dict(copy.deepcopy(init_state))

    train_loader = make_train_loader(id_train_set)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=config.lr,
        momentum=0.9,
        weight_decay=config.weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config.epochs,
    )

    criterion_ce = nn.CrossEntropyLoss()

    miner, metric_loss_fn = build_metric_loss(base)

    if base == "ce":
        metric_weight = 0.0
    else:
        metric_weight = config.metric_weights.get(base, 0.0)

    os.makedirs(config.output_dir, exist_ok=True)
    log_path = os.path.join(
        config.output_dir,
        f"logs_{exp_name}_seed{config.seed}.csv",
    )

    header = [
        "epoch",
        "total_loss",
        "ce_loss",
        "metric_loss",
        "outlier_loss",
        "train_acc",
    ]

    model.train()

    with CSVLogger(log_path, header) as logger:
        for epoch in range(config.epochs):
            total_loss = 0.0
            total_ce = 0.0
            total_metric = 0.0
            total_outlier = 0.0

            correct = 0
            total = 0

            for data, target in train_loader:
                data = data.to(config.device, non_blocking=True)
                target = target.to(config.device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                if base != "ce":
                    logits, embeddings = model(data, return_emb=True)
                else:
                    logits = model(data)
                    embeddings = None

                loss_ce = criterion_ce(logits, target)
                loss = loss_ce

                metric_val = 0.0
                outlier_val = 0.0

                # ------------------------------
                # Metric-learning loss
                # ------------------------------
                if base != "ce":
                    if base == "triplet":
                        indices_tuple = miner(embeddings, target)

                        if indices_tuple is None or all(len(x) == 0 for x in indices_tuple):
                            loss_metric = embeddings.sum() * 0.0
                        else:
                            loss_metric = metric_loss_fn(
                                embeddings,
                                target,
                                indices_tuple,
                            )
                    else:
                        loss_metric = metric_loss_fn(embeddings, target)

                    if torch.isfinite(loss_metric):
                        loss = loss + metric_weight * loss_metric
                        metric_val = loss_metric.item()

                # ------------------------------
                # Synthetic outlier exposure loss
                # ------------------------------
                if use_outliers and config.outlier_batch_size > 0:
                    outlier_data = generate_synthetic_outliers(
                        config.outlier_batch_size,
                        config.device,
                    )

                    # Use eval mode for outlier forward so BatchNorm running stats
                    # are not polluted by synthetic outliers.
                    was_training = model.training
                    model.eval()
                    outlier_logits = model(outlier_data)
                    loss_out = energy_outlier_loss(
                        logits,
                        outlier_logits,
                        T=config.energy_temperature,
                    )
                    if was_training:
                        model.train()

                    if torch.isfinite(loss_out):
                        loss = loss + config.outlier_weight * loss_out
                        outlier_val = loss_out.item()

                if not torch.isfinite(loss):
                    raise RuntimeError(f"Non-finite loss encountered in {exp_name}.")

                loss.backward()
                optimizer.step()

                batch_size = data.size(0)

                total_loss += loss.item() * batch_size
                total_ce += loss_ce.item() * batch_size
                total_metric += metric_val * batch_size
                total_outlier += outlier_val * batch_size

                preds = logits.argmax(dim=1)
                correct += (preds == target).sum().item()
                total += batch_size

            scheduler.step()

            avg_loss = total_loss / max(total, 1)
            avg_ce = total_ce / max(total, 1)
            avg_metric = total_metric / max(total, 1)
            avg_outlier = total_outlier / max(total, 1)
            acc = 100.0 * correct / max(total, 1)

            logger.log([
                epoch + 1,
                f"{avg_loss:.6f}",
                f"{avg_ce:.6f}",
                f"{avg_metric:.6f}",
                f"{avg_outlier:.6f}",
                f"{acc:.2f}",
            ])

            print(
                f"[{exp_name}] Epoch {epoch + 1:02d}/{config.epochs} | "
                f"Loss: {avg_loss:.4f} | "
                f"CE: {avg_ce:.4f} | "
                f"Metric: {avg_metric:.4f} | "
                f"Outlier: {avg_outlier:.4f} | "
                f"Acc: {acc:.2f}%"
            )

    if config.save_models:
        model_path = os.path.join(
            config.output_dir,
            f"model_{exp_name}_seed{config.seed}.pth",
        )
        torch.save(model.state_dict(), model_path)
        print(f"Saved model: {model_path}")

    return model


# ==========================================================
# FEATURE EXTRACTION
# ==========================================================
def get_embeddings_and_logits(model, loader):
    model.eval()

    all_embs = []
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for data, target in loader:
            data = data.to(config.device, non_blocking=True)

            logits, emb = model(data, return_emb=True)

            all_embs.append(emb.cpu())
            all_logits.append(logits.cpu())

            if isinstance(target, torch.Tensor):
                all_labels.append(target.cpu())
            else:
                all_labels.append(torch.tensor(target))

    all_embs = torch.cat(all_embs, dim=0)
    all_logits = torch.cat(all_logits, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    return all_embs, all_logits, all_labels


# ==========================================================
# OOD SCORING FUNCTIONS
# ==========================================================
def msp_ood_score(logits):
    """
    Maximum Softmax Probability.
    Higher score means more likely OOD.
    """
    probs = F.softmax(logits, dim=1)
    max_probs = probs.max(dim=1).values
    return (1.0 - max_probs).numpy()


def energy_ood_score(logits, T=1.0):
    """
    Energy score:
        E(x) = -logsumexp(logits / T)

    Higher energy means more likely OOD.
    """
    return (-torch.logsumexp(logits / T, dim=1)).numpy()


def compute_prototypes(train_emb, train_labels, num_classes):
    train_labels = train_labels.long()
    prototypes = []

    for c in range(num_classes):
        mask = train_labels == c

        if mask.sum() == 0:
            proto = torch.zeros(train_emb.size(1), dtype=train_emb.dtype)
        else:
            proto = train_emb[mask].mean(dim=0)

        proto = F.normalize(proto, dim=0, eps=1e-8)
        prototypes.append(proto)

    return torch.stack(prototypes)


def prototype_ood_score(emb, prototypes):
    """
    Cosine distance to nearest class prototype.
    Higher score means more likely OOD.
    """
    emb = F.normalize(emb, p=2, dim=1)
    prototypes = F.normalize(prototypes, p=2, dim=1)

    sim = emb @ prototypes.t()
    max_sim = sim.max(dim=1).values

    return (1.0 - max_sim).numpy()


def fit_mahalanobis(train_emb, train_labels, num_classes, reg=1e-3):
    """
    Fits class means and a tied covariance matrix.
    """
    train_labels = train_labels.long()
    dim = train_emb.size(1)

    means = []
    covs = []

    for c in range(num_classes):
        x = train_emb[train_labels == c]

        if x.size(0) == 0:
            raise ValueError(f"Class {c} has no training samples for Mahalanobis fitting.")

        mu = x.mean(dim=0)
        diff = x - mu.unsqueeze(0)

        if x.size(0) > 1:
            cov = (diff.t() @ diff) / float(x.size(0) - 1)
        else:
            cov = torch.zeros(dim, dim, dtype=train_emb.dtype)

        means.append(mu)
        covs.append(cov)

    tied_cov = torch.stack(covs).mean(dim=0)
    tied_cov = tied_cov + reg * torch.eye(dim, dtype=tied_cov.dtype)

    try:
        precision = torch.linalg.inv(tied_cov)
    except Exception:
        precision = torch.linalg.pinv(tied_cov)

    return torch.stack(means), precision


def mahalanobis_ood_score(emb, means, precision, chunk_size=512):
    """
    Minimum Mahalanobis distance to any ID class mean.
    Higher distance means more likely OOD.
    """
    emb = F.normalize(emb, p=2, dim=1)

    scores = []

    for start in range(0, emb.size(0), chunk_size):
        x = emb[start:start + chunk_size]
        best_dist = None

        for mu in means:
            diff = x - mu.unsqueeze(0)

            d = torch.einsum("ij,jk,ik->i", diff, precision, diff)
            d = torch.clamp(d, min=0.0)

            if best_dist is None:
                best_dist = d
            else:
                best_dist = torch.minimum(best_dist, d)

        scores.append(best_dist)

    return torch.cat(scores).numpy()


def knn_ood_score(emb, train_emb, k=10, chunk_size=512):
    """
    kNN distance in cosine embedding space.

    We use:
        score = 1 - mean top-k cosine similarity

    Higher score means more likely OOD.
    """
    emb = F.normalize(emb, p=2, dim=1)
    train_emb = F.normalize(train_emb, p=2, dim=1)

    k = min(k, train_emb.size(0))
    train_t = train_emb.t()

    scores = []

    for start in range(0, emb.size(0), chunk_size):
        x = emb[start:start + chunk_size]

        sim = x @ train_t
        topk_sim = sim.topk(k, dim=1).values

        score = 1.0 - topk_sim.mean(dim=1)
        scores.append(score)

    return torch.cat(scores).numpy()


# ==========================================================
# METRICS
# ==========================================================
def compute_binary_metrics(id_scores, ood_scores):
    """
    Labels:
        0 = ID
        1 = OOD

    Scores:
        higher means more likely OOD
    """
    y_true = np.concatenate([
        np.zeros(len(id_scores)),
        np.ones(len(ood_scores)),
    ])

    scores = np.concatenate([id_scores, ood_scores])
    scores = np.nan_to_num(scores, nan=1e6, posinf=1e6, neginf=-1e6)

    try:
        auroc = roc_auc_score(y_true, scores)
    except ValueError:
        auroc = float("nan")

    try:
        fpr, tpr, _ = roc_curve(y_true, scores)
        idx = np.searchsorted(tpr, 0.95)
        idx = min(idx, len(fpr) - 1)
        fpr95 = fpr[idx]
    except Exception:
        fpr95 = float("nan")

    return auroc * 100.0, fpr95 * 100.0


# ==========================================================
# MODEL EVALUATION
# ==========================================================
def evaluate_model(
    exp_name,
    model,
    proto_loader,
    id_test_loader,
    ood_loader,
    ood_name,
    return_embeddings=False,
):
    train_emb, train_logits, train_labels = get_embeddings_and_logits(
        model,
        proto_loader,
    )

    id_emb, id_logits, id_labels = get_embeddings_and_logits(
        model,
        id_test_loader,
    )

    ood_emb, ood_logits, _ = get_embeddings_and_logits(
        model,
        ood_loader,
    )

    id_preds = id_logits.argmax(dim=1)
    id_acc = 100.0 * (id_preds == id_labels).float().mean().item()

    methods = config.score_methods

    prototypes = None
    means = None
    precision = None

    if "Prototype" in methods:
        prototypes = compute_prototypes(
            train_emb,
            train_labels,
            num_classes=len(config.id_classes),
        )

    if "Mahalanobis" in methods:
        means, precision = fit_mahalanobis(
            train_emb,
            train_labels,
            num_classes=len(config.id_classes),
            reg=config.mahalanobis_reg,
        )

    rows = []

    for method in methods:
        if method == "MSP":
            id_scores = msp_ood_score(id_logits)
            ood_scores = msp_ood_score(ood_logits)

        elif method == "Energy":
            id_scores = energy_ood_score(id_logits, T=config.energy_temperature)
            ood_scores = energy_ood_score(ood_logits, T=config.energy_temperature)

        elif method == "Prototype":
            if prototypes is None:
                prototypes = compute_prototypes(
                    train_emb,
                    train_labels,
                    num_classes=len(config.id_classes),
                )

            id_scores = prototype_ood_score(id_emb, prototypes)
            ood_scores = prototype_ood_score(ood_emb, prototypes)

        elif method == "Mahalanobis":
            if means is None or precision is None:
                means, precision = fit_mahalanobis(
                    train_emb,
                    train_labels,
                    num_classes=len(config.id_classes),
                    reg=config.mahalanobis_reg,
                )

            id_scores = mahalanobis_ood_score(id_emb, means, precision)
            ood_scores = mahalanobis_ood_score(ood_emb, means, precision)

        elif method == "kNN":
            id_scores = knn_ood_score(
                id_emb,
                train_emb,
                k=config.knn_k,
            )
            ood_scores = knn_ood_score(
                ood_emb,
                train_emb,
                k=config.knn_k,
            )

        else:
            continue

        auroc, fpr95 = compute_binary_metrics(id_scores, ood_scores)

        rows.append({
            "method": method,
            "auroc": auroc,
            "fpr95": fpr95,
        })

    metrics = pd.DataFrame(rows)

    metrics["experiment"] = exp_name
    metrics["ood_dataset"] = ood_name
    metrics["id_acc"] = id_acc

    metrics = metrics[[
        "experiment",
        "ood_dataset",
        "method",
        "id_acc",
        "auroc",
        "fpr95",
    ]]

    artifacts = {
        "id_acc": id_acc,
    }

    if return_embeddings:
        artifacts["id_emb"] = id_emb
        artifacts["ood_emb"] = ood_emb

    return metrics, artifacts


# ==========================================================
# UMAP VISUALIZATION
# ==========================================================
def visualize_umap(id_emb, ood_emb, title, filename):
    if not UMAP_AVAILABLE:
        print("UMAP is not installed. Skipping UMAP visualization.")
        return

    if torch.is_tensor(id_emb):
        id_emb = id_emb.numpy()

    if torch.is_tensor(ood_emb):
        ood_emb = ood_emb.numpy()

    rng = np.random.RandomState(config.seed)

    if (
        config.umap_max_points is not None
        and config.umap_max_points > 0
        and len(id_emb) > config.umap_max_points
    ):
        idx = rng.choice(len(id_emb), size=config.umap_max_points, replace=False)
        id_emb = id_emb[idx]

    if (
        config.umap_max_points is not None
        and config.umap_max_points > 0
        and len(ood_emb) > config.umap_max_points
    ):
        idx = rng.choice(len(ood_emb), size=config.umap_max_points, replace=False)
        ood_emb = ood_emb[idx]

    all_emb = np.vstack([id_emb, ood_emb])
    all_labels = np.concatenate([
        np.zeros(len(id_emb)),
        np.ones(len(ood_emb)),
    ])

    print(f"Running UMAP for: {title}")

    reducer = umap.UMAP(
        random_state=config.seed,
        n_epochs=config.umap_n_epochs,
        metric="cosine",
        n_jobs=1,
    )

    emb_2d = reducer.fit_transform(all_emb)

    plt.figure(figsize=(8, 6))

    mask_id = all_labels == 0
    mask_ood = all_labels == 1

    plt.scatter(
        emb_2d[mask_id, 0],
        emb_2d[mask_id, 1],
        c="tab:blue",
        marker="o",
        alpha=0.5,
        s=18,
        label="In-Distribution",
    )

    plt.scatter(
        emb_2d[mask_ood, 0],
        emb_2d[mask_ood, 1],
        c="tab:red",
        marker="x",
        alpha=0.6,
        s=35,
        label="Out-of-Distribution",
    )

    plt.legend(fontsize=11)
    plt.title(title, fontsize=13)
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    os.makedirs(config.output_dir, exist_ok=True)
    save_path = os.path.join(config.output_dir, filename)
    plt.savefig(save_path, dpi=200)
    plt.close()

    print(f"Saved UMAP: {save_path}")


# ==========================================================
# MAIN
# ==========================================================
def main():
    set_seeds(config.seed)
    os.makedirs(config.output_dir, exist_ok=True)

    print("=" * 70)
    print("OOD EXPERIMENT")
    print("=" * 70)
    print(f"Device: {config.device}")
    print(f"ID classes: {config.id_classes}")
    print(f"Requested OOD dataset: {config.ood_dataset}")
    print(f"Experiments: {config.experiments}")
    print(f"Score methods: {config.score_methods}")
    print("=" * 70)

    id_train_aug, id_train_clean, id_test, ood_test, ood_name = build_datasets()

    print(f"Actual OOD dataset used: {ood_name}")
    print(f"ID train augmented samples: {len(id_train_aug)}")
    print(f"ID train clean samples: {len(id_train_clean)}")
    print(f"ID test samples: {len(id_test)}")
    print(f"OOD test samples: {len(ood_test)}")

    proto_loader = make_eval_loader(id_train_clean)
    id_test_loader = make_eval_loader(id_test)
    ood_loader = make_eval_loader(ood_test)

    # Create one shared initialization for fair comparison.
    set_seeds(config.seed)
    init_model = ResNetEmbedder(num_classes=len(config.id_classes)).to(config.device)
    init_state = copy.deepcopy(init_model.state_dict())
    del init_model

    all_metrics = []

    for exp_name in config.experiments:
        print("\n" + "=" * 70)
        print(f"RUNNING EXPERIMENT: {exp_name}")
        print("=" * 70)

        model = train_one_experiment(
            exp_name=exp_name,
            init_state=init_state,
            id_train_set=id_train_aug,
        )

        need_embeddings = (
            config.run_umap
            and UMAP_AVAILABLE
            and exp_name in config.umap_experiments
        )

        metrics, artifacts = evaluate_model(
            exp_name=exp_name,
            model=model,
            proto_loader=proto_loader,
            id_test_loader=id_test_loader,
            ood_loader=ood_loader,
            ood_name=ood_name,
            return_embeddings=need_embeddings,
        )

        all_metrics.append(metrics)

        print("\nMetrics:")
        print(metrics.to_string(index=False))

        if need_embeddings:
            visualize_umap(
                id_emb=artifacts["id_emb"],
                ood_emb=artifacts["ood_emb"],
                title=f"{exp_name} vs {ood_name}",
                filename=f"umap_{exp_name}_{ood_name}_seed{config.seed}.png",
            )

        del model
        del artifacts

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if len(all_metrics) > 0:
        results = pd.concat(all_metrics, ignore_index=True)

        results_path = os.path.join(
            config.output_dir,
            f"ood_metrics_seed{config.seed}.csv",
        )

        results.to_csv(results_path, index=False)

        print("\n" + "=" * 70)
        print("FINAL RESULTS")
        print("=" * 70)
        print(results.to_string(index=False))
        print(f"\nSaved metrics to: {results_path}")
    else:
        print("No experiments were run.")


if __name__ == "__main__":
    main()

OOD EXPERIMENT
Device: cuda
ID classes: [0, 1, 2, 3, 4]
Requested OOD dataset: svhn
Experiments: ['ce', 'triplet', 'supcon', 'triplet_outliers']
Score methods: ['MSP', 'Energy', 'Prototype', 'Mahalanobis', 'kNN']


100%|██████████| 170M/170M [21:46<00:00, 131kB/s] 
100%|██████████| 64.3M/64.3M [00:02<00:00, 30.6MB/s]


Actual OOD dataset used: svhn
ID train augmented samples: 25000
ID train clean samples: 25000
ID test samples: 5000
OOD test samples: 5000

RUNNING EXPERIMENT: ce
[ce] Epoch 01/20 | Loss: 1.5021 | CE: 1.5021 | Metric: 0.0000 | Outlier: 0.0000 | Acc: 43.94%
[ce] Epoch 02/20 | Loss: 1.0046 | CE: 1.0046 | Metric: 0.0000 | Outlier: 0.0000 | Acc: 58.24%
[ce] Epoch 03/20 | Loss: 0.8795 | CE: 0.8795 | Metric: 0.0000 | Outlier: 0.0000 | Acc: 64.82%
[ce] Epoch 04/20 | Loss: 0.7581 | CE: 0.7581 | Metric: 0.0000 | Outlier: 0.0000 | Acc: 70.60%
[ce] Epoch 05/20 | Loss: 0.6603 | CE: 0.6603 | Metric: 0.0000 | Outlier: 0.0000 | Acc: 74.99%
[ce] Epoch 06/20 | Loss: 0.5719 | CE: 0.5719 | Metric: 0.0000 | Outlier: 0.0000 | Acc: 78.63%
[ce] Epoch 07/20 | Loss: 0.4959 | CE: 0.4959 | Metric: 0.0000 | Outlier: 0.0000 | Acc: 81.64%
[ce] Epoch 08/20 | Loss: 0.4357 | CE: 0.4357 | Metric: 0.0000 | Outlier: 0.0000 | Acc: 83.78%
[ce] Epoch 09/20 | Loss: 0.3853 | CE: 0.3853 | Metric: 0.0000 | Outlier: 0.0000 | Acc